<a href="https://colab.research.google.com/github/Kanakgoyal27/Healthsystem/blob/main/Ai_work_planner.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [11]:
# Install required libraries

!pip install flask pandas numpy scikit-learn flask-cors

In [12]:
# Import libraries

import os
import sqlite3
import pandas as pd
import numpy as np

from datetime import datetime, date, timedelta

from flask import Flask, request, jsonify, render_template
from flask_cors import CORS

from sklearn.preprocessing import LabelEncoder

print("Libraries imported successfully!")

Libraries imported successfully!


In [13]:
# Create project structure

folders = [
    "ai_work_planner",
    "ai_work_planner/ai",
    "ai_work_planner/database",
    "ai_work_planner/templates",
    "ai_work_planner/static",
    "ai_work_planner/static/css",
    "ai_work_planner/static/js",
    "ai_work_planner/data"
]

for folder in folders:
    os.makedirs(folder, exist_ok=True)

print("Project folders created successfully!")

Project folders created successfully!


In [14]:
# Database location

DB_PATH = "ai_work_planner/database/planner.db"

conn = sqlite3.connect(DB_PATH)

cursor = conn.cursor()

cursor.execute("""
CREATE TABLE IF NOT EXISTS tasks (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    title TEXT NOT NULL,
    description TEXT,
    deadline TEXT,
    priority TEXT,
    estimated_hours REAL,
    status TEXT DEFAULT 'Pending',
    created_at TEXT
)
""")

conn.commit()
conn.close()

print("Database created successfully!")

Database created successfully!


In [15]:
def add_task(title, description, deadline, priority, estimated_hours):
    conn = sqlite3.connect(DB_PATH)
    cursor = conn.cursor()
    created_at = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    status = "Pending"

    cursor.execute("""
        INSERT INTO tasks (title, description, deadline, priority, estimated_hours, status, created_at)
        VALUES (?, ?, ?, ?, ?, ?, ?)
    """, (title, description, deadline, priority, estimated_hours, status, created_at))

    conn.commit()
    conn.close()
    print(f"Task '{title}' added successfully!")

add_task(
    title="Complete Python Assignment",
    description="Finish the machine learning assignment",
    deadline="2026-09-30",
    priority="High",
    estimated_hours=3
)

add_task(
    title="Prepare Presentation",
    description="Prepare project presentation",
    deadline="2026-10-02",
    priority="Medium",
    estimated_hours=2
)

add_task(
    title="Read Research Paper",
    description="Read AI research paper",
    deadline="2026-10-05",
    priority="Low",
    estimated_hours=1
)

Task 'Complete Python Assignment' added successfully!
Task 'Prepare Presentation' added successfully!
Task 'Read Research Paper' added successfully!


In [16]:
def get_tasks():

    conn = sqlite3.connect(DB_PATH)

    df = pd.read_sql_query(
        "SELECT * FROM tasks",
        conn
    )

    conn.close()

    return df


tasks = get_tasks()

display(tasks)

,id,title,description,deadline,priority,estimated_hours,status,created_at
0,1,Complete Python Assignment,Finish the machine learning assignment,2026-09-30,High,3.0,Pending,2026-09-29 10:55:49
1,2,Prepare Presentation,Prepare project presentation,2026-10-02,Medium,2.0,Pending,2026-09-29 10:55:49
2,3,Read Research Paper,Read AI research paper,2026-10-05,Low,1.0,Pending,2026-09-29 10:55:49
3,4,Complete Python Assignment,Finish the machine learning assignment,2026-09-30,High,3.0,Pending,2026-09-29 10:58:51
4,5,Prepare Presentation,Prepare project presentation,2026-10-02,Medium,2.0,Pending,2026-09-29 10:58:51
5,6,Read Research Paper,Read AI research paper,2026-10-05,Low,1.0,Pending,2026-09-29 10:58:51


In [17]:
def calculate_priority_score(task):

    score = 0

    priority_scores = {
        "Low": 1,
        "Medium": 2,
        "High": 3
    }

    score += priority_scores.get(task["priority"], 2) * 10

    if task["deadline"]:

        deadline = datetime.strptime(
            task["deadline"],
            "%Y-%m-%d"
        ).date()

        days_left = (deadline - date.today()).days

        if days_left < 0:
            score += 50
        elif days_left == 0:
            score += 40
        elif days_left == 1:
            score += 30
        elif days_left <= 3:
            score += 20
        else:
            score += 5

    if task["estimated_hours"] >= 4:
        score += 5

    return score

In [18]:
def generate_work_plan():

    tasks = get_tasks()

    pending_tasks = tasks[
        tasks["status"] == "Pending"
    ].copy()

    if pending_tasks.empty:
        print("No pending tasks!")
        return pending_tasks

    pending_tasks["AI_Score"] = pending_tasks.apply(
        calculate_priority_score,
        axis=1
    )

    pending_tasks = pending_tasks.sort_values(
        by="AI_Score",
        ascending=False
    )

    return pending_tasks

In [19]:
plan = generate_work_plan()

display(plan)

,id,title,description,deadline,priority,estimated_hours,status,created_at,AI_Score
0,1,Complete Python Assignment,Finish the machine learning assignment,2026-09-30,High,3.0,Pending,2026-09-29 10:55:49,60
3,4,Complete Python Assignment,Finish the machine learning assignment,2026-09-30,High,3.0,Pending,2026-09-29 10:58:51,60
1,2,Prepare Presentation,Prepare project presentation,2026-10-02,Medium,2.0,Pending,2026-09-29 10:55:49,40
4,5,Prepare Presentation,Prepare project presentation,2026-10-02,Medium,2.0,Pending,2026-09-29 10:58:51,40
2,3,Read Research Paper,Read AI research paper,2026-10-05,Low,1.0,Pending,2026-09-29 10:55:49,15
5,6,Read Research Paper,Read AI research paper,2026-10-05,Low,1.0,Pending,2026-09-29 10:58:51,15
